<h1>Chapter 8 - Semantic Search and Retrieval-Augmented Generation</h1>
<h3>📝 Practice Notebook — Book: <em>Hands-On Large Language Models</em></h3>
<p><em>Work through each TODO block to re-implement the full pipeline from scratch.</em></p>

---

**Topics covered:**
1. Dense Retrieval with Cohere Embeddings + FAISS
2. Sparse Retrieval with BM25
3. Cross-encoder Reranking
4. RAG with Cohere API
5. Local RAG with LlamaCpp + LangChain


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [1]:
# %%capture
%pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1
%pip install llama-cpp-python==0.2.78  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## IMPORTANT: Make sure to restart the session after installing the packages above.

  Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (114 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.0 MB/s eta 0:00:00
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached propcache-0.4.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 9.8 MB/s eta 0:00:00
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 5.5 MB/s eta 0:00:00
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached jsonpoi

# Dense Retrieval Example


## 1. Getting the text archive and chunking it


In [ ]:
import cohere

# Paste your API key here. Remember to not share publicly
api_key = ''

# Create and retrieve a Cohere API key from os.cohere.ai
co = cohere.Client(api_key)

In [ ]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

## 2. Embedding the Text Chunks


In [ ]:
import numpy as np

# TODO: Use co.embed() to get embeddings for all `texts`
#   - input_type should be 'search_document'
#   - Convert the result to a numpy array and store in `embeds`
#   - Print embeds.shape

# YOUR CODE HERE


## 3. Building The Search Index


In [ ]:
import faiss

# TODO: Build a FAISS IndexFlatL2 index
#   - Get `dim` from embeds.shape[1]
#   - Create index = faiss.IndexFlatL2(dim)
#   - Add embeds (cast to float32) to the index

# YOUR CODE HERE


## 4. Search the index


In [ ]:
import pandas as pd

def search(query, number_of_results=3):
    # TODO: Implement semantic search using Cohere embeddings + FAISS
    # Steps:
    #   1. Embed the query with co.embed() using input_type='search_query'
    #   2. Search the FAISS index for nearest neighbors
    #   3. Return a DataFrame with columns: 'texts', 'distance'
    #      (hint: use texts_np = np.array(texts) then index with similar_item_ids)

    # YOUR CODE HERE
    pass


In [ ]:
query = "how precise was the science"
results = search(query)
results

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    # TODO: Tokenize text for BM25
    # Steps:
    #   1. Split on whitespace and lowercase each token
    #   2. Strip punctuation from each token
    #   3. Keep only tokens that are non-empty and not in ENGLISH_STOP_WORDS
    # Return a list of clean tokens

    # YOUR CODE HERE
    pass


In [ ]:
from tqdm import tqdm

# TODO: Build the tokenized corpus and initialise BM25
#   1. Create an empty list tokenized_corpus
#   2. Loop over texts (use tqdm) and append bm25_tokenizer(passage)
#   3. Create bm25 = BM25Okapi(tokenized_corpus)

# YOUR CODE HERE


In [ ]:
def keyword_search(query, top_k=3, num_candidates=15):
    # TODO: Implement BM25 keyword search
    # Steps:
    #   1. Get BM25 scores: bm25.get_scores(bm25_tokenizer(query))
    #   2. Use np.argpartition to find top `num_candidates` indices
    #   3. Build a list of dicts {'corpus_id': idx, 'score': score}
    #   4. Sort descending by score and print top_k results

    # YOUR CODE HERE
    pass


In [ ]:
keyword_search(query = "how precise was the science")

## Caveats of Dense Retrieval


In [ ]:
query = "What is the mass of the moon?"
results = search(query)
results

# Reranking Example


In [ ]:
query = "how precise was the science"

# TODO: Use co.rerank() to rerank all `texts` for this query
#   - top_n=3, return_documents=True
#   - Store in `results` and print results.results

# YOUR CODE HERE


In [ ]:
# TODO: Iterate over results.results and print:
#   idx, result.relevance_score, result.document.text

# YOUR CODE HERE


In [ ]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    # TODO: Implement hybrid BM25 + Cohere reranking pipeline
    # Steps:
    #   1. Get BM25 hits for `num_candidates` results (same as keyword_search)
    #   2. Collect the text for each BM25 hit into a list `docs`
    #   3. Call co.rerank() on those docs with the query
    #   4. Print both the BM25 top-3 and the reranked top-3

    # YOUR CODE HERE
    pass


In [ ]:
keyword_and_reranking_search(query = "how precise was the science")

# Retrieval-Augmented Generation

## Example: Grounded Generation with an LLM API


In [ ]:
query = "income generated"

# TODO: Implement RAG with the Cohere API
# Steps:
#   1. Retrieve relevant docs using search(query)
#   2. Convert the results['texts'] column into a list of dicts: [{'text': t}, ...]
#   3. Call co.chat(message=query, documents=docs_dict)
#   4. Print response.text

# YOUR CODE HERE


In [ ]:
# TODO: Inspect the full `response` object
# (just print / display it)
# YOUR CODE HERE


In [ ]:
# TODO: Access and print response.citations
# YOUR CODE HERE


## Example: RAG with Local Models


### Loading the Generation Model


In [ ]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf

In [ ]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-q4.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

### Loading the Embedding Model

In [ ]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding Model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5'
)

### Preparing the Vector Database

In [ ]:
from langchain.vectorstores import FAISS

# TODO: Create a LangChain FAISS vector store from `texts`
#   Use FAISS.from_texts(texts, embedding_model)
#   Store in variable `db`

# YOUR CODE HERE


### The RAG Prompt


In [ ]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA

# TODO: Build a RAG pipeline with LangChain
# Steps:
#   1. Create a PromptTemplate with variables 'context' and 'question'
#      Use a Phi-3 compatible chat template format:
#        <|user|>\nRelevant info:\n{context}\nAnswer: {question}<|end|>\n<|assistant|>
#   2. Build a RetrievalQA chain with:
#      - llm=llm, chain_type='stuff'
#      - retriever=db.as_retriever()
#      - chain_type_kwargs={'prompt': prompt}
#      - verbose=True
#   Store in variable `rag`

# YOUR CODE HERE


In [ ]:
rag.invoke('Income generated')